# AIC 2026 — ASR ingestion · ChunkFormer RNNT Large

**Input:** `oh-i-ace/aic-videos`  
**Output:** `aqpahm/aic2026-asr-chunkformer-rnnt-large`  
**Primary:** `khanhld/chunkformer-rnnt-large-vie`; recovery dùng Whisper khi cần.

Notebook nguồn chạy độc lập trên **Google Colab hoặc Kaggle**. Bật GPU, Internet
và secret `HF_TOKEN`. Completion được audit theo video thực sự có trong archive.


In [ ]:
!pip install -q chunkformer==1.2.2 "huggingface_hub>=0.34,<2" pyarrow


In [ ]:
import gc
import importlib.metadata
import json
import os
import sys
import tempfile
import re
import shutil
import subprocess
import time
import unicodedata
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from queue import Queue
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download

INPUT_REPO = "oh-i-ace/aic-videos"
OUTPUT_REPO = "aqpahm/aic2026-asr-chunkformer-rnnt-large"
MODEL_ID = "khanhld/chunkformer-rnnt-large-vie"
ARCHIVE_NAMES = (
    "Videos_L21_a.zip", "Videos_L22_a.zip", "Videos_L23_a.zip",
    "Videos_L24_a.zip", "Videos_L25_a.zip", "Videos_L26_a.zip",
    "Videos_L26_b.zip", "Videos_L26_c.zip", "Videos_L26_d.zip",
    "Videos_L26_e.zip", "Videos_L27_a.zip", "Videos_L28_a.zip",
    "Videos_L29_a.zip", "Videos_L30_a.zip",
)
UPLOAD_BATCH_VIDEOS = 10
STRICT_REMOTE_VALIDATION = False  # True re-downloads and validates every marker
WORKER_INDEX = 0
WORKER_COUNT = 1
if not 0 <= WORKER_INDEX < WORKER_COUNT:
    raise ValueError("WORKER_INDEX must satisfy 0 <= index < count")
SELECTED_ARCHIVES = [
    name for index, name in enumerate(ARCHIVE_NAMES)
    if index % WORKER_COUNT == WORKER_INDEX
]

def detect_runtime():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if RUNTIME == "colab":
        from google.colab import userdata
        value = userdata.get(name)
    elif RUNTIME == "kaggle":
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    if not value:
        raise RuntimeError(
            f"Missing {name}. Add it as a Colab/Kaggle secret and enable notebook access."
        )
    return value


def hosted_work_root(job_name):
    if RUNTIME == "colab":
        return Path("/content") / job_name
    if RUNTIME == "kaggle":
        return Path("/kaggle/temp") / job_name
    return Path(tempfile.gettempdir()) / job_name

RUNTIME = detect_runtime()
ROOT = hosted_work_root("aic-asr")
DOWNLOAD_DIR = ROOT / "download"
EXTRACT_DIR = ROOT / "extracted"
OUTPUT_DIR = ROOT / "output"
for path in (DOWNLOAD_DIR, EXTRACT_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

HF_TOKEN = read_secret("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
account = api.whoami()
print("Runtime:", RUNTIME)
print("Hugging Face:", account["name"])
api.create_repo(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    private=True,
    exist_ok=True,
)
INPUT_REVISION = api.dataset_info(INPUT_REPO).sha
MODEL_REVISION = api.model_info(MODEL_ID).sha
CHUNKFORMER_VERSION = importlib.metadata.version("chunkformer")
print("ChunkFormer package:", CHUNKFORMER_VERSION)
print("Model revision:", MODEL_REVISION)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in the Colab/Kaggle runtime before running this job")
GPU_IDS = list(range(torch.cuda.device_count()))
DEVICES = [f"cuda:{device_id}" for device_id in GPU_IDS]
CPU_THREADS_PER_WORKER = max(1, min(4, (os.cpu_count() or 2) // len(GPU_IDS)))
torch.set_num_threads(CPU_THREADS_PER_WORKER)
print("GPU workers:", len(GPU_IDS))
for device_id in GPU_IDS:
    properties = torch.cuda.get_device_properties(device_id)
    print(f"  cuda:{device_id}: {properties.name}, {properties.total_memory / 1024**3:.1f} GiB")


In [ ]:
from chunkformer import ChunkFormerModel

torch.backends.cuda.matmul.allow_tf32 = True
models = []
for device in DEVICES:
    current_model = ChunkFormerModel.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION
    ).to(device)
    current_model.eval()
    models.append(current_model)
print(f"Loaded {len(models)} ChunkFormer replica(s):", MODEL_ID)


In [ ]:
def timestamp_seconds(value):
    if isinstance(value, (int, float)):
        return float(value)
    value = str(value).strip().replace(",", ".")
    parts = value.split(":")
    if len(parts) == 4:
        hours, minutes, seconds, milliseconds = parts
        return int(hours) * 3600 + int(minutes) * 60 + int(seconds) + int(milliseconds) / 1000
    if len(parts) == 3:
        hours, minutes, seconds = parts
        return int(hours) * 3600 + int(minutes) * 60 + float(seconds)
    return float(value)


def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text)).strip()
    return re.sub(r"\s+", " ", text).casefold()


def normalize_no_accent(text):
    text = normalize_text(text).replace("đ", "d")
    return "".join(
        char for char in unicodedata.normalize("NFD", text)
        if unicodedata.category(char) != "Mn"
    )


def extract_audio(video_path, audio_path):
    command = [
        "ffmpeg", "-nostdin", "-y", "-loglevel", "error",
        "-i", str(video_path), "-vn", "-ac", "1", "-ar", "16000",
        "-c:a", "pcm_s16le", str(audio_path),
    ]
    subprocess.run(command, check=True)


def probe_audio(video_path):
    completed = subprocess.run(
        [
            "ffprobe", "-v", "error", "-select_streams", "a:0",
            "-show_entries", "stream=codec_name,sample_rate,channels:format=duration",
            "-of", "json", str(video_path),
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    info = json.loads(completed.stdout)
    if not info.get("streams"):
        raise RuntimeError(f"{video_path.stem} has no audio stream")
    duration = float(info.get("format", {}).get("duration", 0))
    if duration <= 0:
        raise RuntimeError(f"{video_path.stem} has invalid audio duration")
    stream = info["streams"][0]
    return {
        "duration_sec": duration,
        "codec": stream.get("codec_name"),
        "sample_rate": int(stream.get("sample_rate", 0)),
        "channels": int(stream.get("channels", 0)),
    }


def transcribe_video(video_path, active_model):
    audio_path = ROOT / f"{video_path.stem}.wav"
    audio_info = probe_audio(video_path)
    print("  audio:", audio_info)
    try:
        extract_audio(video_path, audio_path)
        # The reference inference path defaults to FP32. Keep the correctness
        # baseline in FP32; lower precision is a separate measured optimization.
        total_batch_duration = 300
        while True:
            try:
                with torch.inference_mode():
                    decoded = active_model.endless_decode(
                        audio_path=str(audio_path),
                        chunk_size=64,
                        left_context_size=128,
                        right_context_size=128,
                        total_batch_duration=total_batch_duration,
                        return_timestamps=True,
                    )
                break
            except (torch.OutOfMemoryError, RuntimeError) as error:
                is_oom = isinstance(error, torch.OutOfMemoryError) or "out of memory" in str(error).lower()
                if not is_oom or total_batch_duration <= 75:
                    raise
                total_batch_duration = max(75, total_batch_duration // 2)
                print(f"  CUDA OOM; retry ASR with total_batch_duration={total_batch_duration}")
                gc.collect()
                torch.cuda.empty_cache()
    finally:
        audio_path.unlink(missing_ok=True)

    print("  raw decode groups:", len(decoded), "sample:", decoded[:1])
    rows = []
    for index, item in enumerate(decoded):
        text = str(item.get("decode", item.get("text", ""))).strip()
        if not text:
            continue
        start_sec = timestamp_seconds(item["start"])
        end_sec = timestamp_seconds(item["end"])
        rows.append({
            "video_id": video_path.stem,
            "segment_id": f"{video_path.stem}__asr_{index:05d}",
            "start_sec": start_sec,
            "end_sec": end_sec,
            "timestamp_sec": (start_sec + end_sec) / 2,
            "raw_text": unicodedata.normalize("NFC", text),
            "normalized_text": normalize_text(text),
            "normalized_no_accent": normalize_no_accent(text),
            "language": "vi",
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        })
    if not rows:
        raise RuntimeError(
            "ChunkFormer returned no speech segments; do not publish a success marker"
        )
    previous_start = -1.0
    for row in rows:
        if row["start_sec"] < previous_start:
            raise RuntimeError("ASR timestamps are not monotonic")
        if row["start_sec"] < 0 or row["end_sec"] <= row["start_sec"]:
            raise RuntimeError(f"Invalid ASR interval: {row}")
        if row["end_sec"] > audio_info["duration_sec"] + 2:
            raise RuntimeError(f"ASR timestamp exceeds audio duration: {row}")
        previous_start = row["start_sec"]
    return rows, audio_info


def upload_files(paths, message):
    operations = []
    for local_path in paths:
        relative = local_path.relative_to(OUTPUT_DIR).as_posix()
        operations.append(CommitOperationAdd(path_in_repo=f"data/{relative}", path_or_fileobj=str(local_path)))
    for attempt in range(6):
        try:
            api.create_commit(
                repo_id=OUTPUT_REPO,
                repo_type="dataset",
                operations=operations,
                commit_message=message,
                token=HF_TOKEN,
            )
            return
        except Exception as error:
            if attempt == 5:
                raise
            delay = min(120, 10 * (2 ** attempt))
            print(f"Upload lỗi: {type(error).__name__}. Thử lại sau {delay}s")
            time.sleep(delay)


def remote_video_completed(video_id):
    """Fast resume by artifact trio; optionally validate marker provenance."""
    level = video_id.split("_")[0]
    prefix = f"data/{level}/{video_id}"
    marker_remote = f"{prefix}/_ASR_SUCCESS.json"
    required = (marker_remote, f"{prefix}/asr.parquet", f"{prefix}/asr_chunks.parquet")
    if not all(path in repo_files for path in required):
        return False
    if not STRICT_REMOTE_VALIDATION:
        return True
    try:
        marker_path = hf_hub_download(
            repo_id=OUTPUT_REPO,
            filename=marker_remote,
            repo_type="dataset",
            token=HF_TOKEN,
            force_download=True,
        )
        payload = json.loads(Path(marker_path).read_text(encoding="utf-8"))
        primary_matches = (
            payload.get("model_id") == MODEL_ID
            and payload.get("model_revision") == MODEL_REVISION
        ) or (
            payload.get("primary_model_id") == MODEL_ID
            and payload.get("primary_model_revision") == MODEL_REVISION
        )
        status = payload.get("status", "success")
        has_text = (
            int(payload.get("segments", 0)) > 0
            and int(payload.get("search_chunks", 0)) > 0
        )
        confirmed_empty = (
            status == "no_detected_speech"
            and int(payload.get("segments", 0)) == 0
        )
        return (
            primary_matches
            and int(payload.get("schema_version", 0)) >= 3
            and (has_text or confirmed_empty)
        )
    except Exception as error:
        print(f"Remote marker invalid for {video_id}: {error!r}")
        return False

def make_search_chunks(rows, target_duration=25.0, max_duration=35.0, max_gap=3.0):
    """Merge adjacent raw segments while retaining one-segment overlap."""
    chunks = []
    start_index = 0
    while start_index < len(rows):
        end_index = start_index
        while end_index + 1 < len(rows):
            current = rows[end_index]
            following = rows[end_index + 1]
            gap = following["start_sec"] - current["end_sec"]
            proposed_duration = following["end_sec"] - rows[start_index]["start_sec"]
            if gap > max_gap or proposed_duration > max_duration:
                break
            end_index += 1
            if rows[end_index]["end_sec"] - rows[start_index]["start_sec"] >= target_duration:
                break

        selected = rows[start_index : end_index + 1]
        start_sec = selected[0]["start_sec"]
        end_sec = selected[-1]["end_sec"]
        raw_text = " ".join(item["raw_text"] for item in selected).strip()
        chunks.append({
            "video_id": selected[0]["video_id"],
            "chunk_id": f"{selected[0]['video_id']}__asr_chunk_{len(chunks):05d}",
            "start_sec": start_sec,
            "end_sec": end_sec,
            "timestamp_sec": (start_sec + end_sec) / 2,
            "raw_text": raw_text,
            "normalized_text": normalize_text(raw_text),
            "normalized_no_accent": normalize_no_accent(raw_text),
            "source_segment_start": selected[0]["segment_id"],
            "source_segment_end": selected[-1]["segment_id"],
            "language": "vi",
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        })
        start_index = end_index if end_index > start_index else start_index + 1
    return chunks


RAW_COLUMNS = [
    "video_id", "segment_id", "start_sec", "end_sec", "timestamp_sec",
    "raw_text", "normalized_text", "normalized_no_accent", "language",
    "model_id", "model_revision",
]
CHUNK_COLUMNS = [
    "video_id", "chunk_id", "start_sec", "end_sec", "timestamp_sec",
    "raw_text", "normalized_text", "normalized_no_accent",
    "source_segment_start", "source_segment_end", "language",
    "model_id", "model_revision",
]


def is_video_member(member):
    return re.fullmatch(r"L\d{2}_V\d{3}\.mp4", Path(member).name, re.IGNORECASE) is not None


def write_video_output(video_id, rows, audio_info):
    level = video_id.split("_")[0]
    directory = OUTPUT_DIR / level / video_id
    directory.mkdir(parents=True, exist_ok=True)
    parquet_path = directory / "asr.parquet"
    chunks_path = directory / "asr_chunks.parquet"
    marker_path = directory / "_ASR_SUCCESS.json"
    chunks = make_search_chunks(rows)
    pd.DataFrame(rows, columns=RAW_COLUMNS).to_parquet(parquet_path, index=False)
    pd.DataFrame(chunks, columns=CHUNK_COLUMNS).to_parquet(chunks_path, index=False)
    marker = {
        "video_id": video_id,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "chunkformer_version": CHUNKFORMER_VERSION,
        "segments": len(rows),
        "search_chunks": len(chunks),
        "audio": audio_info,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "schema_version": 3,
    }
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
    return [parquet_path, chunks_path, marker_path], chunks
# This source is embedded in the Kaggle notebook; it is not a local download job.
import sys

DOWNLOAD_IDLE_SECONDS = 180
DOWNLOAD_ATTEMPT_SECONDS = 1800
DOWNLOAD_ATTEMPTS = 5


class ArchiveDownloadError(RuntimeError):
    pass


def download_progress_bytes(directory, filename):
    # Serial downloads only. Include the HF .incomplete file, retained on retry.
    paths = list(directory.rglob("*.incomplete")) + [directory / filename]
    total = 0
    for path in paths:
        try:
            total += path.stat().st_size
        except FileNotFoundError:
            pass
    return total


def download_archive(filename):
    """Isolate the transfer so a stalled SDK call can actually be stopped."""
    child_source = '''
import os
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id=os.environ["ASR_INPUT_REPO"],
    filename=os.environ["ASR_ARCHIVE_NAME"],
    revision=os.environ["ASR_INPUT_REVISION"],
    repo_type="dataset", token=os.environ["HF_TOKEN"],
    local_dir=os.environ["ASR_DOWNLOAD_DIR"],
)
'''
    child_env = os.environ.copy()
    child_env.update({
        "HF_TOKEN": HF_TOKEN,
        "ASR_INPUT_REPO": INPUT_REPO,
        "ASR_ARCHIVE_NAME": filename,
        "ASR_INPUT_REVISION": INPUT_REVISION,
        "ASR_DOWNLOAD_DIR": str(DOWNLOAD_DIR),
        # Set BEFORE importing huggingface_hub in the fresh child process.
        "HF_HUB_DISABLE_XET": "1",
        "HF_HUB_ENABLE_HF_TRANSFER": "0",
        "HF_HUB_DOWNLOAD_TIMEOUT": "30",
        "HF_HUB_ETAG_TIMEOUT": "30",
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
        "PYTHONUNBUFFERED": "1",
    })
    destination = DOWNLOAD_DIR / filename
    log_path = DOWNLOAD_DIR / (filename + ".download.log")
    for attempt in range(1, DOWNLOAD_ATTEMPTS + 1):
        previous = download_progress_bytes(DOWNLOAD_DIR, filename)
        started = last_change = time.monotonic()
        print(f"Download {filename}: attempt {attempt}/{DOWNLOAD_ATTEMPTS}; "
              f"retained data {previous / 1e9:.2f} GB", flush=True)
        reason = "transfer failed"
        with log_path.open("w", encoding="utf-8") as log:
            process = subprocess.Popen(
                [sys.executable, "-u", "-c", child_source],
                env=child_env, stdout=log, stderr=subprocess.STDOUT,
            )
            try:
                while True:
                    try:
                        return_code = process.wait(timeout=15)
                        break
                    except subprocess.TimeoutExpired:
                        now = time.monotonic()
                        current = download_progress_bytes(DOWNLOAD_DIR, filename)
                        if current != previous:
                            last_change = now
                        print(f"  {filename}: cached {current / 1e9:.2f} GB; "
                              f"change {(current - previous) / 1e6:.1f} MB/15s; "
                              f"idle {now - last_change:.0f}s", flush=True)
                        previous = current
                        if now - last_change >= DOWNLOAD_IDLE_SECONDS:
                            reason = "no download progress for 180 seconds"
                            return_code = None
                            break
                        if now - started >= DOWNLOAD_ATTEMPT_SECONDS:
                            reason = "attempt reached 30 minutes; reconnecting"
                            return_code = None
                            break
            finally:
                # Also stop the child when the user presses Cancel Run.
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
        if return_code == 0 and destination.is_file():
            print(f"Download complete: {filename}; checking ZIP CRC...", flush=True)
            try:
                with zipfile.ZipFile(destination) as archive:
                    bad_member = archive.testzip()
                if bad_member is not None:
                    raise zipfile.BadZipFile(f"CRC mismatch: {bad_member}")
            except zipfile.BadZipFile as error:
                # Do not extract or silently discard corrupted data.
                raise ArchiveDownloadError(
                    f"{filename}: ZIP validation failed ({error}). File retained for diagnosis."
                ) from error
            return destination
        if return_code is not None:
            reason = f"download process exited with code {return_code}"
        print(f"  Retry needed: {reason}. Partial download and cache retained.", flush=True)
        if attempt < DOWNLOAD_ATTEMPTS:
            time.sleep(min(60, 10 * attempt))
    raise ArchiveDownloadError(
        f"{filename}: failed after {DOWNLOAD_ATTEMPTS} attempts. "
        f"Partial data retained in {DOWNLOAD_DIR}. Rerun in this session to resume. "
        f"Diagnostic log: {log_path} (do not publish without checking for signed URLs)."
    )


In [ ]:
# Full L21-L30 ingestion. Safe to rerun after an interrupted Colab/Kaggle session.
repo_files = set(api.list_repo_files(repo_id=OUTPUT_REPO, repo_type="dataset", token=HF_TOKEN))
candidate_ids = {
    path.split("/")[-2]
    for path in repo_files
    if path.endswith("/_ASR_SUCCESS.json")
}
completed_ids = set()
for number, video_id in enumerate(sorted(candidate_ids), 1):
    if remote_video_completed(video_id):
        completed_ids.add(video_id)
    if number % 100 == 0:
        print(f"Checked {number}/{len(candidate_ids)} remote markers")
print("Valid remote videos:", len(completed_ids))

pending_files = []
pending_ids = []
all_seen_ids = set()
failures = []
archive_failures = []
qa_rows = []
new_count = 0
skip_count = 0
device_pool = Queue()
for model_index in range(len(models)):
    device_pool.put(model_index)


def flush_pending():
    global pending_files, pending_ids
    if not pending_files:
        return
    upload_files(
        pending_files,
        f"Add ChunkFormer RNNT ASR for {pending_ids[0]} through {pending_ids[-1]}",
    )
    completed_ids.update(pending_ids)
    print(f"✅ Uploaded one commit for {len(pending_ids)} videos")
    for video_id in pending_ids:
        shutil.rmtree(OUTPUT_DIR / video_id.split("_")[0] / video_id, ignore_errors=True)
    pending_files = []
    pending_ids = []


def process_member(archive_path, archive_name, member):
    model_index = device_pool.get()
    device_id = GPU_IDS[model_index]
    video_id = Path(member).stem
    video_path = EXTRACT_DIR / Path(member).name
    try:
        with zipfile.ZipFile(archive_path) as archive:
            with archive.open(member) as source, video_path.open("wb") as target:
                shutil.copyfileobj(source, target, length=1024 * 1024)
        started = time.perf_counter()
        torch.cuda.reset_peak_memory_stats(device_id)
        rows, audio_info = transcribe_video(video_path, models[model_index])
        files, chunks = write_video_output(video_id, rows, audio_info)
        torch.cuda.synchronize(device_id)
        elapsed = time.perf_counter() - started
        return {
            "archive": archive_name,
            "video_id": video_id,
            "files": files,
            "rows": rows,
            "segments": len(rows),
            "chunks": len(chunks),
            "elapsed_sec": elapsed,
            "peak_vram_gib": torch.cuda.max_memory_allocated(device_id) / 1024**3,
            "device": f"cuda:{device_id}",
        }
    finally:
        video_path.unlink(missing_ok=True)
        gc.collect()
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()
        device_pool.put(model_index)


for archive_index, archive_name in enumerate(SELECTED_ARCHIVES, 1):
    archive_path = None
    print("=" * 72)
    print(f"[{archive_index}/{len(SELECTED_ARCHIVES)}] {archive_name}")
    try:
        archive_path = download_archive(archive_name)
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            members = sorted(
                member for member in archive.namelist()
                if is_video_member(member)
            )
        if not members:
            raise RuntimeError(f"No organizer videos found in {archive_name}")
        video_ids = [Path(member).stem for member in members]
        if len(video_ids) != len(set(video_ids)):
            raise RuntimeError(f"Duplicate video IDs in {archive_name}")
        all_seen_ids.update(video_ids)
        pending_members = [member for member in members if Path(member).stem not in completed_ids]
        skipped_here = len(members) - len(pending_members)
        skip_count += skipped_here
        print(
            f"Videos found: {len(members)}; skip valid: {skipped_here}; "
            f"pending: {len(pending_members)}; GPU workers: {len(models)}"
        )

        with ThreadPoolExecutor(max_workers=len(models)) as executor:
            futures = {
                executor.submit(process_member, archive_path, archive_name, member): member
                for member in pending_members
            }
            for position, future in enumerate(as_completed(futures), 1):
                member = futures[future]
                video_id = Path(member).stem
                try:
                    result = future.result()
                    pending_files.extend(result["files"])
                    pending_ids.append(video_id)
                    new_count += 1
                    if len(qa_rows) < 50:
                        qa_rows.extend(result["rows"][: 50 - len(qa_rows)])
                    print(
                        f"[{position}/{len(futures)}] ✅ {video_id}: "
                        f"{result['segments']} segments, {result['chunks']} chunks, "
                        f"{result['elapsed_sec']:.1f}s, {result['peak_vram_gib']:.2f} GiB, "
                        f"{result['device']}"
                    )
                    if len(pending_ids) >= UPLOAD_BATCH_VIDEOS:
                        flush_pending()
                except Exception as error:
                    failures.append({
                        "archive": archive_name,
                        "video_id": video_id,
                        "error": repr(error),
                    })
                    print(f"[{position}/{len(futures)}] ❌ {video_id}: {error!r}")
        flush_pending()
    except ArchiveDownloadError:
        flush_pending()
        raise
    except Exception as error:
        archive_failures.append({"archive": archive_name, "error": repr(error)})
        print("Archive error:", repr(error))
        flush_pending()
    finally:
        shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
        if archive_path is not None:
            archive_path.unlink(missing_ok=True)
        gc.collect()
        for device_id in GPU_IDS:
            with torch.cuda.device(device_id):
                torch.cuda.empty_cache()

print("=" * 72)
print("GPU workers:", len(models))
print("Archives selected:", len(SELECTED_ARCHIVES))
print("Videos seen:", len(all_seen_ids))
print("Newly processed:", new_count)
print("Skipped valid:", skip_count)
print("Video failures:", len(failures))
print("Archive failures:", len(archive_failures))
if failures:
    print(json.dumps(failures, ensure_ascii=False, indent=2))
if archive_failures:
    print(json.dumps(archive_failures, ensure_ascii=False, indent=2))
fallback_failures = [
    item for item in failures
    if "ChunkFormer returned no speech segments" in item["error"]
]
hard_failures = [item for item in failures if item not in fallback_failures]
print("Queued for Whisper fallback:", len(fallback_failures))
if hard_failures or archive_failures:
    raise RuntimeError(
        "The run has non-ASR-empty failures. Rerun; valid remote videos will be skipped."
    )


In [ ]:
"""Recover empty ASR results after the preceding ingestion cell."""

import gc
import importlib.metadata
import json
import shutil
import subprocess
import sys
import unicodedata
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch

try:
    from faster_whisper import WhisperModel
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "faster-whisper>=1.1,<2",
    ])
    from faster_whisper import WhisperModel


MANUAL_RECOVERY_TARGETS = {}
RECOVERY_TARGETS = {}
for item in globals().get("fallback_failures", []):
    RECOVERY_TARGETS.setdefault(item["archive"], set()).add(item["video_id"])
for archive_name, video_ids in MANUAL_RECOVERY_TARGETS.items():
    RECOVERY_TARGETS.setdefault(archive_name, set()).update(video_ids)
print("Whisper fallback targets:", sum(map(len, RECOVERY_TARGETS.values())))

FALLBACK_MODEL_ID = "Systran/faster-whisper-large-v3"
FALLBACK_DEVICE_INDEX = 1 if torch.cuda.device_count() > 1 else 0
FALLBACK_COMPUTE_TYPE = "float16"

whisper_model = None


def get_whisper_model():
    global whisper_model
    if whisper_model is not None:
        return whisper_model
    for current_model in models:
        current_model.to("cpu")
    gc.collect()
    for device_id in GPU_IDS:
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()
    print("Loading fallback on GPU", FALLBACK_DEVICE_INDEX)
    whisper_model = WhisperModel(
        FALLBACK_MODEL_ID,
        device="cuda",
        device_index=FALLBACK_DEVICE_INDEX,
        compute_type=FALLBACK_COMPUTE_TYPE,
    )
    return whisper_model

def retry_primary_transcription(video_path):
    global whisper_model
    # Release Whisper before restoring ChunkFormer, including on a single T4.
    whisper_model = None
    gc.collect()
    for device_id in GPU_IDS:
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()
    active_model = models[0].to(DEVICES[0])
    return transcribe_video(video_path, active_model)


def whisper_rows(video_path, use_vad=True):
    """Transcribe once and materialize the lazy faster-whisper generator."""
    kwargs = {
        "beam_size": 5,
        "task": "transcribe",
        "language": None,
        "condition_on_previous_text": False,
        "vad_filter": use_vad,
    }
    if use_vad:
        kwargs["vad_parameters"] = {
            "min_silence_duration_ms": 500,
            "speech_pad_ms": 400,
        }

    segments, info = get_whisper_model().transcribe(str(video_path), **kwargs)
    segments = list(segments)  # Inference happens here.
    language = info.language or "und"
    revision = f"faster-whisper:{importlib.metadata.version('faster-whisper')}"
    rows = []
    for segment in segments:
        text = unicodedata.normalize("NFC", str(segment.text)).strip()
        if not text:
            continue
        # When VAD is disabled, reject segments that Whisper itself considers
        # likely silence or very low confidence to reduce silence hallucinations.
        if not use_vad:
            if float(getattr(segment, "no_speech_prob", 1.0)) >= 0.60:
                continue
            if float(getattr(segment, "avg_logprob", -99.0)) <= -1.0:
                continue
        start_sec = max(0.0, float(segment.start))
        end_sec = float(segment.end)
        if end_sec <= start_sec:
            continue
        rows.append({
            "video_id": video_path.stem,
            "segment_id": f"{video_path.stem}__asr_{len(rows):05d}",
            "start_sec": start_sec,
            "end_sec": end_sec,
            "timestamp_sec": (start_sec + end_sec) / 2,
            "raw_text": text,
            "normalized_text": normalize_text(text),
            "normalized_no_accent": normalize_no_accent(text),
            "language": language,
            "model_id": FALLBACK_MODEL_ID,
            "model_revision": revision,
        })
    return rows, {
        "language": language,
        "language_probability": float(info.language_probability or 0.0),
        "vad_filter": use_vad,
    }


def recovery_chunks(rows):
    chunks = []
    start_index = 0
    while start_index < len(rows):
        end_index = start_index
        while end_index + 1 < len(rows):
            current = rows[end_index]
            following = rows[end_index + 1]
            gap = following["start_sec"] - current["end_sec"]
            duration = following["end_sec"] - rows[start_index]["start_sec"]
            if gap > 3.0 or duration > 35.0:
                break
            end_index += 1
            if rows[end_index]["end_sec"] - rows[start_index]["start_sec"] >= 25.0:
                break
        selected = rows[start_index:end_index + 1]
        text = " ".join(row["raw_text"] for row in selected).strip()
        start_sec = selected[0]["start_sec"]
        end_sec = selected[-1]["end_sec"]
        chunks.append({
            "video_id": selected[0]["video_id"],
            "chunk_id": f"{selected[0]['video_id']}__asr_chunk_{len(chunks):05d}",
            "start_sec": start_sec,
            "end_sec": end_sec,
            "timestamp_sec": (start_sec + end_sec) / 2,
            "raw_text": text,
            "normalized_text": normalize_text(text),
            "normalized_no_accent": normalize_no_accent(text),
            "source_segment_start": selected[0]["segment_id"],
            "source_segment_end": selected[-1]["segment_id"],
            "language": selected[0]["language"],
            "model_id": selected[0]["model_id"],
            "model_revision": selected[0]["model_revision"],
        })
        start_index = end_index if end_index > start_index else start_index + 1
    return chunks


def write_recovery_output(video_id, rows, audio_info, method, diagnostics):
    directory = OUTPUT_DIR / video_id.split("_")[0] / video_id
    shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True, exist_ok=True)
    chunks = recovery_chunks(rows)
    parquet_path = directory / "asr.parquet"
    chunks_path = directory / "asr_chunks.parquet"
    marker_path = directory / "_ASR_SUCCESS.json"
    pd.DataFrame(rows, columns=RAW_COLUMNS).to_parquet(parquet_path, index=False)
    pd.DataFrame(chunks, columns=CHUNK_COLUMNS).to_parquet(chunks_path, index=False)

    status = "success" if rows else "no_detected_speech"
    output_model = rows[0]["model_id"] if rows else FALLBACK_MODEL_ID
    output_revision = rows[0]["model_revision"] if rows else (
        f"faster-whisper:{importlib.metadata.version('faster-whisper')}"
    )
    marker = {
        "video_id": video_id,
        "status": status,
        "model_id": output_model,
        "model_revision": output_revision,
        "primary_model_id": MODEL_ID,
        "primary_model_revision": MODEL_REVISION,
        "fallback_attempted": method != "chunkformer_retry",
        "recovery_method": method,
        "segments": len(rows),
        "search_chunks": len(chunks),
        "audio": audio_info,
        "diagnostics": diagnostics,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "schema_version": 3,
    }
    marker_path.write_text(
        json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return [parquet_path, chunks_path, marker_path]


recovery_files = []
recovery_report = []
recovered_ids = set()

for archive_name, target_ids in RECOVERY_TARGETS.items():
    archive_path = download_archive(archive_name)
    try:
        with zipfile.ZipFile(archive_path) as archive:
            members = {
                Path(name).stem: name
                for name in archive.namelist()
                if Path(name).stem in target_ids
                and Path(name).suffix.lower() in {".mp4", ".mkv", ".avi", ".mov"}
            }
            absent = sorted(target_ids - set(members))
            if absent:
                raise RuntimeError(f"Missing target videos in {archive_name}: {absent}")

            for number, video_id in enumerate(sorted(target_ids), 1):
                member = members[video_id]
                video_path = EXTRACT_DIR / Path(member).name
                video_path.parent.mkdir(parents=True, exist_ok=True)
                print(f"[{archive_name} {number}/{len(target_ids)}] {video_id}")
                try:
                    with archive.open(member) as source, video_path.open("wb") as target:
                        shutil.copyfileobj(source, target, length=1024 * 1024)
                    audio_info = probe_audio(video_path)

                    # Required order: retry the primary model first.
                    try:
                        rows, audio_info = retry_primary_transcription(video_path)
                        method = "chunkformer_retry"
                        diagnostics = {"chunkformer": "non_empty"}
                    except RuntimeError as error:
                        if "no speech segments" not in str(error):
                            raise
                        print("  ChunkFormer is still empty; trying Whisper large-v3 + VAD")
                        rows, whisper_info = whisper_rows(video_path, use_vad=True)
                        method = "whisper_vad"
                        diagnostics = {
                            "chunkformer": "empty",
                            "whisper_vad": whisper_info,
                        }
                        if not rows:
                            print("  Whisper + VAD is empty; checking once without VAD")
                            rows, whisper_info_no_vad = whisper_rows(video_path, use_vad=False)
                            method = "whisper_no_vad" if rows else "verified_empty_after_fallback"
                            diagnostics["whisper_no_vad"] = whisper_info_no_vad

                    recovery_files.extend(
                        write_recovery_output(
                            video_id, rows, audio_info, method, diagnostics
                        )
                    )
                    recovered_ids.add(video_id)
                    recovery_report.append({
                        "video_id": video_id,
                        "status": "success" if rows else "no_detected_speech",
                        "method": method,
                        "segments": len(rows),
                        "preview": " ".join(row["raw_text"] for row in rows[:2])[:240],
                    })
                    print("  ->", recovery_report[-1])
                finally:
                    video_path.unlink(missing_ok=True)
                    gc.collect()
                    torch.cuda.empty_cache()
    finally:
        archive_path.unlink(missing_ok=True)

expected = set().union(*RECOVERY_TARGETS.values())
if recovered_ids != expected:
    raise RuntimeError(f"Recovery incomplete: {sorted(expected - recovered_ids)}")

# Publish recovery artifacts together after all target videos succeed.
if recovery_files:
    upload_files(recovery_files, "Recover empty ChunkFormer ASR videos with Whisper fallback")
    completed_ids.update(recovered_ids)
    report_df = pd.DataFrame(recovery_report).sort_values("video_id")
    display(report_df)
    print("Recovered with text:", int((report_df.status == "success").sum()))
    print("Confirmed no detected speech:", int((report_df.status == "no_detected_speech").sum()))
else:
    print("No Whisper fallback work is needed.")
missing_ids = all_seen_ids - completed_ids
if missing_ids:
    raise RuntimeError(f"ASR completion audit failed: {sorted(missing_ids)}")
print("Dataset:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")
